In [1]:
# =============================================================================
# Phase 5 — Hive Data Warehouse : Gold → Hive External Tables
# Craigslist Used Vehicle Dataset
# =============================================================================

import logging
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
log = logging.getLogger("VehicleHive")


In [2]:
# =============================================================================
# CONFIG
# =============================================================================

HIVE_DB   = "vehicle_market"
GOLD_BASE = "hdfs:///user/vehicle_market/gold"

GOLD_TABLES = {
    "kpi_summary"           : f"{GOLD_BASE}/kpi_summary",
    "manufacturer_summary"  : f"{GOLD_BASE}/manufacturer_summary",
    "state_summary"         : f"{GOLD_BASE}/state_summary",
    "monthly_summary"       : f"{GOLD_BASE}/monthly_summary",
    "fuel_summary"          : f"{GOLD_BASE}/fuel_summary",
    "segment_summary"       : f"{GOLD_BASE}/segment_summary",
    "condition_summary"     : f"{GOLD_BASE}/condition_summary",
    "title_summary"         : f"{GOLD_BASE}/title_summary",
    "transmission_summary"  : f"{GOLD_BASE}/transmission_summary",
    "state_manufacturer"    : f"{GOLD_BASE}/state_manufacturer",
    "manufacturer_condition": f"{GOLD_BASE}/manufacturer_condition",
}

In [3]:
# =============================================================================
# 1. SPARK SESSION
# =============================================================================

spark = SparkSession.builder \
    .appName("VehicleMarket_Hive") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

pipeline_start = time.time()
log.info("="*65)
log.info("  Phase 5 — Hive Data Warehouse : Gold → Hive")
log.info("="*65)

26/07/20 11:00:21 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
2026-07-20 11:00:21 | INFO | =================================================================
2026-07-20 11:00:21 | INFO |   Phase 5 — Hive Data Warehouse : Gold → Hive
2026-07-20 11:00:21 | INFO | =================================================================


In [4]:
# =============================================================================
# 2. VALIDATE GOLD PARQUET FILES EXIST IN HDFS
# Check before attempting table creation to fail fast with a clear message
# =============================================================================

log.info("Validating Gold Parquet datasets in HDFS...")

sc   = spark.sparkContext
fs   = sc._jvm.org.apache.hadoop.fs.FileSystem.get(sc._jsc.hadoopConfiguration())
missing_paths = []

for table_name, path in GOLD_TABLES.items():
    hdfs_path = sc._jvm.org.apache.hadoop.fs.Path(path)
    if not fs.exists(hdfs_path):
        missing_paths.append((table_name, path))
        log.error(f"  ✗ Missing: {table_name} → {path}")
    else:
        log.info(f"  ✓ Found  : {table_name}")

if missing_paths:
    raise FileNotFoundError(
        f"The following Gold datasets are missing from HDFS: "
        f"{[t for t, _ in missing_paths]}. "
        f"Re-run Phase 4 before proceeding."
    )

log.info("  All Gold datasets validated successfully.")

2026-07-20 11:00:21 | INFO | Validating Gold Parquet datasets in HDFS...
2026-07-20 11:00:21 | INFO |   ✓ Found  : kpi_summary
2026-07-20 11:00:21 | INFO |   ✓ Found  : manufacturer_summary
2026-07-20 11:00:21 | INFO |   ✓ Found  : state_summary
2026-07-20 11:00:21 | INFO |   ✓ Found  : monthly_summary
2026-07-20 11:00:21 | INFO |   ✓ Found  : fuel_summary
2026-07-20 11:00:21 | INFO |   ✓ Found  : segment_summary
2026-07-20 11:00:21 | INFO |   ✓ Found  : condition_summary
2026-07-20 11:00:21 | INFO |   ✓ Found  : title_summary
2026-07-20 11:00:21 | INFO |   ✓ Found  : transmission_summary
2026-07-20 11:00:21 | INFO |   ✓ Found  : state_manufacturer
2026-07-20 11:00:21 | INFO |   ✓ Found  : manufacturer_condition
2026-07-20 11:00:21 | INFO |   All Gold datasets validated successfully.


In [5]:
# =============================================================================
# 3. CREATE HIVE DATABASE
# =============================================================================

log.info(f"Creating Hive database: {HIVE_DB}...")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {HIVE_DB}")
spark.sql(f"USE {HIVE_DB}")
log.info(f"  Using database: {HIVE_DB}")

2026-07-20 11:00:21 | INFO | Creating Hive database: vehicle_market...
26/07/20 11:00:22 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/07/20 11:00:22 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
Loading class `com.mysql.jdbc.Driver'. This is deprecated. The new driver class is `com.mysql.cj.jdbc.Driver'. The driver is automatically registered via the SPI and manual loading of the driver class is generally unnecessary.
26/07/20 11:00:23 WARN ObjectStore: Failed to get database vehicle_market, returning NoSuchObjectException
26/07/20 11:00:23 WARN ObjectStore: Failed to get database vehicle_market, returning NoSuchObjectException
26/07/20 11:00:23 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException
26/07/20 11:00:23 WARN ObjectStore: Failed to get database vehicle_market, returning NoSuchObjectException
2026-07-20 11:00:23 | INFO |   Using database: vehicle_market


In [6]:
# =============================================================================
# 4. HELPER — Load Gold Parquet and derive schema for Hive DDL
# External tables point to Gold HDFS locations — no data is copied
# =============================================================================

def get_hive_type(spark_type):
    """Map Spark SQL type to Hive-compatible DDL type."""
    type_map = {
        "LongType"    : "BIGINT",
        "IntegerType" : "INT",
        "DoubleType"  : "DOUBLE",
        "FloatType"   : "FLOAT",
        "StringType"  : "STRING",
        "BooleanType" : "BOOLEAN",
        "TimestampType": "TIMESTAMP",
        "DateType"    : "DATE",
    }
    return type_map.get(type(spark_type).__name__, "STRING")


def create_external_table(table_name, hdfs_path):
    """
    Read Gold Parquet, derive schema, drop existing table,
    and create Hive external table pointing to HDFS path.
    """
    df     = spark.read.parquet(hdfs_path)
    schema = df.schema

    col_defs = ",\n  ".join(
        f"`{field.name}` {get_hive_type(field.dataType)}"
        for field in schema.fields
    )

    ddl = f"""
        CREATE EXTERNAL TABLE IF NOT EXISTS {HIVE_DB}.{table_name} (
          {col_defs}
        )
        STORED AS PARQUET
        LOCATION '{hdfs_path}'
    """

    # Drop first to ensure schema stays in sync with Gold on re-runs
    spark.sql(f"DROP TABLE IF EXISTS {HIVE_DB}.{table_name}")
    spark.sql(ddl)
    spark.sql(f"ANALYZE TABLE {HIVE_DB}.{table_name} COMPUTE STATISTICS")

    row_count = spark.sql(
        f"SELECT COUNT(*) AS cnt FROM {HIVE_DB}.{table_name}"
    ).collect()[0]["cnt"]

    return row_count, len(schema.fields)

In [7]:
# =============================================================================
# 5. CREATE ALL HIVE EXTERNAL TABLES
# =============================================================================

log.info("Creating Hive external tables...")

creation_results = {}

for table_name, hdfs_path in GOLD_TABLES.items():
    try:
        row_count, col_count = create_external_table(table_name, hdfs_path)
        creation_results[table_name] = {
            "status"   : "✓",
            "rows"     : row_count,
            "cols"     : col_count,
            "path"     : hdfs_path
        }
        log.info(
            f"  ✓ {table_name:<30} "
            f"rows: {row_count:<8} cols: {col_count}"
        )
    except Exception as e:
        creation_results[table_name] = {
            "status": "✗", "rows": 0, "cols": 0, "path": hdfs_path
        }
        log.error(f"  ✗ {table_name} FAILED: {e}")

tables_created = sum(
    1 for v in creation_results.values() if v["status"] == "✓"
)
tables_failed  = len(GOLD_TABLES) - tables_created

if tables_failed > 0:
    raise RuntimeError(
        f"{tables_failed} Hive table(s) failed to create. "
        f"Review errors above before running analytical queries."
    )

log.info(f"  All {tables_created} Hive external tables created successfully.")

2026-07-20 11:00:23 | INFO | Creating Hive external tables...
26/07/20 11:00:24 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/07/20 11:00:24 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/07/20 11:00:24 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/07/20 11:00:24 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
2026-07-20 11:00:24 | INFO |   ✓ kpi_summary                    rows: 1        cols: 15
2026-07-20 11:00:25 | INFO |   ✓ manufacturer_summary           rows: 43       cols: 15
2026-07-20 11:00:25 | INFO |   ✓ state_summary                  rows: 51       cols: 14
2026-07-20 11:00:25 | INFO |   ✓ monthly_summary                rows: 2        cols: 15
2026-07-20 11:00:26 | INFO |   ✓ fuel_summary                   rows: 6        cols: 11
2026-07-20 11:00:26 | INFO |   ✓ segment_summ

In [8]:
# =============================================================================
# 6. SCHEMA & STORAGE VERIFICATION
# Confirm Hive sees correct columns and HDFS location for each table
# =============================================================================

log.info("Verifying table schemas and storage locations...")

for table_name in GOLD_TABLES:
    # col_count = len(spark.sql(
    #     f"DESCRIBE {HIVE_DB}.{table_name}"
    # ).collect())
    describe_rows = spark.sql(
        f"DESCRIBE {HIVE_DB}.{table_name}"
    ).collect()

    col_count = sum(
        1
        for r in describe_rows
        if r["col_name"] and not r["col_name"].startswith("#")
    )

    location = None
    for row in spark.sql(
        f"DESCRIBE FORMATTED {HIVE_DB}.{table_name}"
    ).collect():
        if "Location" in str(row["col_name"]):
            location = row["data_type"].strip()
            break

    log.info(
        f"  {table_name:<30} cols: {col_count:<5} "
        f"location: {location}"
    )

2026-07-20 11:00:27 | INFO | Verifying table schemas and storage locations...
2026-07-20 11:00:27 | INFO |   kpi_summary                    cols: 15    location: hdfs://127.0.0.1:9000/user/vehicle_market/gold/kpi_summary
2026-07-20 11:00:27 | INFO |   manufacturer_summary           cols: 15    location: hdfs://127.0.0.1:9000/user/vehicle_market/gold/manufacturer_summary
2026-07-20 11:00:28 | INFO |   state_summary                  cols: 14    location: hdfs://127.0.0.1:9000/user/vehicle_market/gold/state_summary
2026-07-20 11:00:28 | INFO |   monthly_summary                cols: 15    location: hdfs://127.0.0.1:9000/user/vehicle_market/gold/monthly_summary
2026-07-20 11:00:28 | INFO |   fuel_summary                   cols: 11    location: hdfs://127.0.0.1:9000/user/vehicle_market/gold/fuel_summary
2026-07-20 11:00:28 | INFO |   segment_summary                cols: 12    location: hdfs://127.0.0.1:9000/user/vehicle_market/gold/segment_summary
2026-07-20 11:00:28 | INFO |   condition_sum

In [9]:
# =============================================================================
# 7. ANALYTICAL SQL QUERIES
# Representative queries validating each table — one per Gold table
# These mirror the Tableau dashboard query patterns
# =============================================================================

log.info("\n" + "="*65)
log.info("  Running Analytical SQL Validation Queries")
log.info("="*65)

query_results = {}

# def run_query(label, sql):
#     """Execute a SQL query, log results, and record pass/fail."""
#     try:
#         result = spark.sql(sql)
#         result.show(10, truncate=False)
#         query_results[label] = "✓ PASSED"
#         log.info(f"  ✓ {label}")
#     except Exception as e:
#         query_results[label] = f"✗ FAILED: {e}"
#         log.error(f"  ✗ {label} — {e}")
def run_query(label, sql, expect_rows=True):
    """Execute a SQL query, log results, and record pass/fail.
    expect_rows=True raises if the result is empty.
    """
    try:
        result = spark.sql(sql)
        row_count = result.count()
        if expect_rows and row_count == 0:
            raise ValueError(f"Query returned 0 rows — possible data issue.")
        result.show(10, truncate=False)
        query_results[label] = f"✓ PASSED ({row_count} rows)"
        log.info(f"  ✓ {label} ({row_count} rows)")
    except Exception as e:
        query_results[label] = f"✗ FAILED: {e}"
        log.error(f"  ✗ {label} — {e}")


# --- Q1: Executive KPI Snapshot (Dashboard 1 KPI cards) ---
run_query(
    "Q01 — Executive KPI Snapshot",
    f"""
    SELECT
        total_listings,
        avg_price,
        median_price,
        avg_mileage,
        avg_vehicle_age,
        total_manufacturers,
        total_states,
        clean_title_pct,
        luxury_pct,
        salvage_pct,
        automatic_pct
    FROM {HIVE_DB}.kpi_summary
    """
)

# --- Q2: Top 10 Manufacturers by Listings (Dashboard 1) ---
run_query(
    "Q02 — Top 10 Manufacturers by Listings",
    f"""
    SELECT
        manufacturer,
        total_listings,
        avg_price,
        median_price,
        avg_vehicle_age,
        is_luxury
    FROM {HIVE_DB}.manufacturer_summary
    ORDER BY total_listings DESC
    LIMIT 10
    """
)

# --- Q3: Top 10 Most Expensive Manufacturers (Dashboard 2) ---
run_query(
    "Q03 — Top 10 Manufacturers by Avg Price",
    f"""
    SELECT
        manufacturer,
        avg_price,
        median_price,
        total_listings,
        is_luxury
    FROM {HIVE_DB}.manufacturer_summary
    WHERE total_listings >= 100
    ORDER BY avg_price DESC
    LIMIT 10
    """
)

# --- Q4: State-wise Average Price Ranking (Dashboard 4) ---
run_query(
    "Q04 — State-wise Average Price Ranking",
    f"""
    SELECT
        state,
        total_listings,
        avg_price,
        median_price,
        avg_mileage,
        avg_vehicle_age,
        luxury_pct
    FROM {HIVE_DB}.state_summary
    ORDER BY avg_price DESC
    LIMIT 15
    """
)

# --- Q5: Monthly Listing Trend (Dashboard 1 trend chart) ---
run_query(
    "Q05 — Monthly Listing Trend",
    f"""
    SELECT
        posting_year,
        posting_month,
        posting_month_name,
        posting_season,
        total_listings,
        avg_price,
        median_price
    FROM {HIVE_DB}.monthly_summary
    ORDER BY posting_year, posting_month
    """
)

# --- Q6: Seasonal Listing Patterns (Dashboard 1) ---
run_query(
    "Q06 — Seasonal Listing Patterns",
    f"""
    SELECT
        posting_season,
        SUM(total_listings)              AS total_listings,
        ROUND(AVG(avg_price), 2)         AS avg_price,
        ROUND(AVG(median_price), 2)      AS avg_median_price
    FROM {HIVE_DB}.monthly_summary
    WHERE posting_season IS NOT NULL
    GROUP BY posting_season
    ORDER BY total_listings DESC
    """
)

# --- Q7: Fuel Type Distribution (Dashboard 1 donut) ---
run_query(
    "Q07 — Fuel Type Distribution",
    f"""
    SELECT
        fuel,
        is_alternative_fuel,
        total_listings,
        listing_pct,
        avg_price,
        median_price,
        avg_mileage
    FROM {HIVE_DB}.fuel_summary
    ORDER BY total_listings DESC
    """
)

# --- Q8: Vehicle Segment Analysis (Dashboard 3 treemap) ---
run_query(
    "Q08 — Vehicle Segment Analysis",
    f"""
    SELECT
        vehicle_segment,
        total_listings,
        listing_pct,
        avg_price,
        median_price,
        avg_mileage,
        avg_vehicle_age,
        clean_title_pct
    FROM {HIVE_DB}.segment_summary
    ORDER BY total_listings DESC
    """
)

# --- Q9: Vehicle Condition vs Price (Dashboard 2 heatmap) ---
run_query(
    "Q09 — Vehicle Condition vs Price",
    f"""
    SELECT
        condition,
        total_listings,
        listing_pct,
        avg_price,
        median_price,
        min_price,
        max_price
    FROM {HIVE_DB}.condition_summary
    ORDER BY avg_price DESC
    """
)

# --- Q10: Title Status Breakdown (Dashboard 3) ---
run_query(
    "Q10 — Title Status Breakdown",
    f"""
    SELECT
        title_status,
        clean_title,
        is_salvage,
        total_listings,
        listing_pct,
        avg_price,
        median_price,
        avg_vehicle_age
    FROM {HIVE_DB}.title_summary
    ORDER BY total_listings DESC
    """
)

# --- Q11: Transmission Type Analysis (Dashboard 2 & 3) ---
run_query(
    "Q11 — Transmission Type Analysis",
    f"""
    SELECT
        transmission,
        total_listings,
        listing_pct,
        avg_price,
        median_price,
        avg_mileage,
        avg_vehicle_age
    FROM {HIVE_DB}.transmission_summary
    ORDER BY total_listings DESC
    """
)

# --- Q12: Top Manufacturer per State (Dashboard 4) ---
run_query(
    "Q12 — Top Manufacturer per State",
    f"""
    SELECT
        state,
        manufacturer,
        total_listings,
        avg_price
    FROM (
        SELECT
            state,
            manufacturer,
            total_listings,
            avg_price,
            ROW_NUMBER() OVER (
                PARTITION BY state ORDER BY total_listings DESC
            ) AS rn
        FROM {HIVE_DB}.state_manufacturer
    ) ranked
    WHERE rn = 1
    ORDER BY total_listings DESC
    LIMIT 15
    """
)

# --- Q13: Manufacturer × Condition Price Matrix (Dashboard 2 heatmap) ---
run_query(
    "Q13 — Manufacturer × Condition Price Matrix",
    f"""
    SELECT
        manufacturer,
        condition,
        total_listings,
        avg_price,
        median_price
    FROM {HIVE_DB}.manufacturer_condition
    WHERE
        manufacturer != 'unknown'
        AND condition  != 'unknown'
        AND total_listings >= 50
    ORDER BY manufacturer, avg_price DESC
    LIMIT 20
    """
)

# --- Q14: Luxury vs Non-Luxury Market Comparison (Dashboard 2) ---
run_query(
    "Q14 — Luxury vs Non-Luxury Market Comparison",
    f"""
    SELECT
        is_luxury,
        COUNT(*)                              AS manufacturer_count,
        SUM(total_listings)                   AS total_listings,
        ROUND(AVG(avg_price), 2)              AS avg_price,
        ROUND(AVG(median_price), 2)           AS avg_median_price,
        ROUND(AVG(avg_mileage), 0)            AS avg_mileage,
        ROUND(AVG(avg_vehicle_age), 1)        AS avg_vehicle_age,
        ROUND(AVG(clean_title_pct), 2)        AS avg_clean_title_pct
    FROM {HIVE_DB}.manufacturer_summary
    GROUP BY is_luxury
    """
)

# --- Q15: Alternative Fuel Market Share by State (Dashboard 4) ---
run_query(
    "Q15 — Alternative Fuel Market Share by State",
    f"""
    SELECT
        state,
        total_listings,
        alt_fuel_pct,
        avg_price
    FROM {HIVE_DB}.state_summary
    WHERE alt_fuel_pct > 0
    ORDER BY alt_fuel_pct DESC
    LIMIT 15
    """
)

2026-07-20 11:00:28 | INFO | 
2026-07-20 11:00:28 | INFO |   Running Analytical SQL Validation Queries
2026-07-20 11:00:28 | INFO | =================================================================
2026-07-20 11:00:29 | INFO |   ✓ Q01 — Executive KPI Snapshot (1 rows)
2026-07-20 11:00:29 | INFO |   ✓ Q02 — Top 10 Manufacturers by Listings (10 rows)


+--------------+---------+------------+-----------+---------------+-------------------+------------+---------------+----------+-----------+-------------+
|total_listings|avg_price|median_price|avg_mileage|avg_vehicle_age|total_manufacturers|total_states|clean_title_pct|luxury_pct|salvage_pct|automatic_pct|
+--------------+---------+------------+-----------+---------------+-------------------+------------+---------------+----------+-----------+-------------+
|242979        |15365.38 |10999       |107376.0   |15.4           |43                 |51          |93.96          |14.97     |3.99       |88.67        |
+--------------+---------+------------+-----------+---------------+-------------------+------------+---------------+----------+-----------+-------------+

+------------+--------------+---------+------------+---------------+---------+
|manufacturer|total_listings|avg_price|median_price|avg_vehicle_age|is_luxury|
+------------+--------------+---------+------------+---------------+---

Java HotSpot(TM) 64-Bit Server VM warning: CodeCache is full. Compiler has been disabled.
Java HotSpot(TM) 64-Bit Server VM warning: Try increasing the code cache size using -XX:ReservedCodeCacheSize=
2026-07-20 11:00:29 | INFO |   ✓ Q03 — Top 10 Manufacturers by Avg Price (10 rows)
2026-07-20 11:00:29 | INFO |   ✓ Q04 — State-wise Average Price Ranking (15 rows)


CodeCache: size=131072Kb used=51930Kb max_used=51930Kb free=79141Kb
 bounds [0x0000000102a38000, 0x0000000105d38000, 0x000000010aa38000]
 total_blobs=17414 nmethods=16279 adapters=1047
 compilation: disabled (not enough contiguous free space left)
+-------------+---------+------------+--------------+---------+
|manufacturer |avg_price|median_price|total_listings|is_luxury|
+-------------+---------+------------+--------------+---------+
|tesla        |42348.19 |39999       |279           |true     |
|porsche      |30447.08 |25900       |784           |true     |
|alfa-romeo   |27952.57 |28163       |207           |false    |
|rover        |26797.06 |22590       |1197          |false    |
|ram          |26562.85 |24999       |8758          |false    |
|gmc          |21063.62 |16499       |8992          |false    |
|mercedes-benz|19461.27 |14999       |6524          |true     |
|audi         |19187.04 |16499       |3534          |true     |
|jaguar       |18049.36 |12995       |716       

2026-07-20 11:00:29 | INFO |   ✓ Q05 — Monthly Listing Trend (2 rows)


...

+---------+--------------+-----------+---------+------------+---------+---------+
|condition|total_listings|listing_pct|avg_price|median_price|min_price|max_price|
+---------+--------------+-----------+---------+------------+---------+---------+
|new      |920           |0.38       |26828.55 |20000       |103      |140000   |
|like new |17421         |7.17       |18451.53 |13995       |100      |144990   |
|unknown  |96433         |39.69      |17729.73 |13988       |100      |150000   |
|excellent|69642         |28.66      |14318.68 |10990       |100      |150000   |
|good     |52284         |21.52      |12557.85 |7997        |100      |150000   |
|fair     |5769          |2.37       |3807.86  |2500        |100      |123456   |
|salvage  |510           |0.21       |3691.53  |1850        |200      |35989    |
+---------+--------------+-----------+---------+------------+---------+---------+

+------------+-----------+----------+--------------+-----------+---------+------------+----------

2026-07-20 11:00:30 | INFO |   ✓ Q12 — Top Manufacturer per State (15 rows)
2026-07-20 11:00:31 | INFO |   ✓ Q13 — Manufacturer × Condition Price Matrix (20 rows)


+-----+------------+--------------+---------+
|state|manufacturer|total_listings|avg_price|
+-----+------------+--------------+---------+
|ca   |ford        |3885          |17952.37 |
|tx   |ford        |3044          |23346.98 |
|fl   |ford        |2995          |19701.03 |
|mi   |ford        |2508          |13779.39 |
|oh   |ford        |1850          |13731.12 |
|or   |ford        |1560          |18814.54 |
|ny   |ford        |1377          |15359.06 |
|wi   |ford        |1268          |14843.78 |
|nc   |ford        |1238          |17661.96 |
|il   |ford        |1200          |15134.27 |
+-----+------------+--------------+---------+
only showing top 10 rows

+------------+---------+--------------+---------+------------+
|manufacturer|condition|total_listings|avg_price|median_price|
+------------+---------+--------------+---------+------------+
|acura       |good     |716           |16627.82 |15000       |
|acura       |like new |127           |13003.39 |11000       |
|acura       |e

2026-07-20 11:00:31 | INFO |   ✓ Q14 — Luxury vs Non-Luxury Market Comparison (2 rows)
2026-07-20 11:00:31 | INFO |   ✓ Q15 — Alternative Fuel Market Share by State (15 rows)


+---------+------------------+--------------+---------+----------------+-----------+---------------+-------------------+
|is_luxury|manufacturer_count|total_listings|avg_price|avg_median_price|avg_mileage|avg_vehicle_age|avg_clean_title_pct|
+---------+------------------+--------------+---------+----------------+-----------+---------------+-------------------+
|true     |14                |36381         |24210.06 |21154.71        |87359.0    |16.4           |94.59              |
|false    |29                |206598        |14733.08 |12034.14        |94416.0    |16.9           |94.65              |
+---------+------------------+--------------+---------+----------------+-----------+---------------+-------------------+

+-----+--------------+------------+---------+
|state|total_listings|alt_fuel_pct|avg_price|
+-----+--------------+------------+---------+
|ms   |694           |16.57       |13018.23 |
|hi   |7             |14.29       |11378.14 |
|az   |5383          |8.99        |16073.95

In [10]:
# =============================================================================
# 8. LIST ALL HIVE TABLES IN DATABASE
# =============================================================================

log.info("Listing all tables in Hive database...")
spark.sql(f"SHOW TABLES IN {HIVE_DB}").show(truncate=False)



2026-07-20 11:00:31 | INFO | Listing all tables in Hive database...


+--------------+----------------------+-----------+
|namespace     |tableName             |isTemporary|
+--------------+----------------------+-----------+
|vehicle_market|condition_summary     |false      |
|vehicle_market|fuel_summary          |false      |
|vehicle_market|kpi_summary           |false      |
|vehicle_market|manufacturer_condition|false      |
|vehicle_market|manufacturer_summary  |false      |
|vehicle_market|monthly_summary       |false      |
|vehicle_market|segment_summary       |false      |
|vehicle_market|state_manufacturer    |false      |
|vehicle_market|state_summary         |false      |
|vehicle_market|title_summary         |false      |
|vehicle_market|transmission_summary  |false      |
+--------------+----------------------+-----------+



In [11]:
# =============================================================================
# 9. WAREHOUSE VALIDATION REPORT
# =============================================================================

pipeline_end   = time.time()
execution_time = round(pipeline_end - pipeline_start, 2)

queries_passed = sum(1 for v in query_results.values() if v.startswith("✓"))
queries_failed = sum(1 for v in query_results.values() if v.startswith("✗"))

log.info("\n" + "="*65)
log.info("      HIVE WAREHOUSE — VALIDATION REPORT")
log.info("="*65)
log.info(f"  Execution Time          : {execution_time}s")
log.info(f"  Hive Database           : {HIVE_DB}")
log.info(f"  Tables Created          : {tables_created} / {len(GOLD_TABLES)}")
log.info(f"  Queries Executed        : {len(query_results)}")
log.info(f"  Queries Passed          : {queries_passed}")
log.info(f"  Queries Failed          : {queries_failed}")
log.info("")
log.info(f"  {'Table':<30} {'Status':>6}  {'Rows':>8}  {'Cols':>6}")
log.info(f"  {'-'*55}")

for table_name, result in creation_results.items():
    log.info(
        f"  {result['status']}  {table_name:<30} "
        f"{result['rows']:>8}  {result['cols']:>6}"
    )

log.info("")
log.info(f"  {'Query':<45} {'Status'}")
log.info(f"  {'-'*60}")

for query_label, status in query_results.items():
    log.info(f"  {query_label:<45} {status}")

log.info("")
# if queries_failed == 0 and tables_failed == 0:
#     log.info("  ✓ Hive Data Warehouse complete.")
#     log.info("  ✓ All tables registered. All queries passed.")
#     log.info("  ✓ Ready for Kafka streaming pipeline.")
# else:
#     log.info("  ⚠ Some tables or queries failed. Review errors above.")
if queries_failed == 0 and tables_failed == 0:
    log.info("  ✓ Hive Data Warehouse complete.")
    log.info("  ✓ External tables registered successfully.")
    log.info("  ✓ Gold datasets are now available for SQL analytics and Tableau.")
else:
    log.info("  ⚠ Some tables or queries failed. Review errors above.")

log.info("="*65)

2026-07-20 11:00:31 | INFO | 
2026-07-20 11:00:31 | INFO |       HIVE WAREHOUSE — VALIDATION REPORT
2026-07-20 11:00:31 | INFO | =================================================================
2026-07-20 11:00:31 | INFO |   Execution Time          : 10.3s
2026-07-20 11:00:31 | INFO |   Hive Database           : vehicle_market
2026-07-20 11:00:31 | INFO |   Tables Created          : 11 / 11
2026-07-20 11:00:31 | INFO |   Queries Executed        : 15
2026-07-20 11:00:31 | INFO |   Queries Passed          : 15
2026-07-20 11:00:31 | INFO |   Queries Failed          : 0
2026-07-20 11:00:31 | INFO | 
2026-07-20 11:00:31 | INFO |   Table                          Status      Rows    Cols
2026-07-20 11:00:31 | INFO |   -------------------------------------------------------
2026-07-20 11:00:31 | INFO |   ✓  kpi_summary                           1      15
2026-07-20 11:00:31 | INFO |   ✓  manufacturer_summary                 43      15
2026-07-20 11:00:31 | INFO |   ✓  state_summary           